In [4]:
import os
from selenium import webdriver
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup
import google.generativeai as genai
import csv
from email.message import EmailMessage 
import smtplib
import ssl

In [9]:
def get_website_data(url):
    driver = webdriver.Chrome()  #Turns on Chrome
    driver.get(url) #gets the page
    driver.implicitly_wait(20)  # Wait for elements to load (optional)
    content=""
    soup = BeautifulSoup(driver.page_source)  
    paragraphs= soup.find_all("p",limit=10) #finds paragraphs and store as list with tags p
    for paragraph in paragraphs:
        content = content+(paragraph.text) #store all contents to content variable
    driver.quit()
    return content

def agency_or_not(content):
    genai.configure(api_key=os.getenv("MY_API_KEY"))  #gets api key
    model = genai.GenerativeModel("gemini-1.5-flash")
    promptbegi="Based on the following website content, does this represent an agency that provides services like web design, web development, SEO, digital marketing, Website creation, Ads Agency? Provide a 'Yes' or 'No' answer. Content: "
    prompt=promptbegi+content
    response = model.generate_content(prompt)  #genrate yes or no based on the prompt
    return (response.text.rstrip())  

def decision_storage(response,agency_name,emailcomp):
    email_sender = 'mozilorassignment123@gmail.com'
    email_password = os.getenv("EMAIL_PWD")
    email_receiver = emailcomp
    company = agency_name
    if response=="Yes":        
        with open("agency_decisions.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([agency_name,"Approved"])
        subject = 'Welcome to CookieYes!'
        body = f"""
                    We are glad to announce your {company} is eligible for our product CookieYes. 
                    Thank You
                """
    else:
        with open("agency_decisions.csv", mode="a", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([agency_name,"Declined"])
        subject = 'Eligibilty Declined!'
        body = f"""
                    We are sorry to announce your {company} is not eligible for our product CookieYes. 
                    Thank You
                """
    em = EmailMessage()
    em['From'] = email_sender
    em['To'] = email_receiver
    em['Subject'] = subject
    em.set_content(body)

    # Add SSL (layer of security)
    context = ssl.create_default_context()

    # Log in and send the email
    with smtplib.SMTP_SSL('smtp.gmail.com', 465, context=context) as smtp:
        smtp.login(email_sender, email_password)
        smtp.sendmail(email_sender, email_receiver, em.as_string())
 

def main():
    # List of websites and their corresponding names
    websites = [
        {"name": "Four By North", "url": "https://fourbynorth.com/","email":"hello@fourbynorth.com"},
        {"name": "Digital Silk", "url": "https://www.digitalsilk.com/","email":"hello@digitalsilk.com"},
        {"name": "Baunfire", "url": "https://www.baunfire.com/","email":"hello@baunfire.com"}
    ]
    with open("agency_decisions.csv", mode="w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["Agency Name", "Decision"])
    for website in websites:
        print(f"Processing {website['name']}...")
        try:
            content = get_website_data(website["url"])
            response = agency_or_not(content)
            print(f"Is the website {website['name']} accepted: {response}")
            decision_storage(response, website["name"],website["email"])
        except Exception as e:
            print(f"An error occurred with {website['name']}: {e}")

        
if __name__ == "__main__":
    main()





Processing Four By North...
Is the website Four By North accepted: Yes
Processing Digital Silk...
Is the website Digital Silk accepted: Yes
Processing Baunfire...
Is the website Baunfire accepted: Yes
